# IOAI — 2024 Summer National Animal Classifier (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CIFAR-10 은 torchvision 으로, 동결 ImageNet ResNet18 가중치는 torchvision 으로 자동 다운로드됩니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Smart Farmer John — Animal Census (Intelligens János gazda)

> **HAIO 2024 — Summer National Finals (CV)**

János photographed his neighbour's farm animals and needs you to **identify each animal and tally the herd**. Decide the species in every image, then report the per-species counts.

**Approach (faithful to the original):** use a **frozen ImageNet-pretrained model** as a fixed feature extractor — *no backbone training* — and put a small classifier on top. Score = **accuracy** over the held-out animal photos.

**Data note:** the original Google-Drive animal photos were deleted by the organisers, so we reconstruct the *same task* (frozen-pretrained animal classification + census) on the **CIFAR-10 animal subset** — 6 species: `bird, cat, deer, dog, frog, horse` (auto-downloaded by torchvision). **Submit** `submission.csv` (`id,label`, label 0–5 in that species order).

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0); device='cuda' if torch.cuda.is_available() else 'cpu'
ANIMALS=[2,3,4,5,6,7]  # CIFAR-10: bird,cat,deer,dog,frog,horse
REMAP={c:i for i,c in enumerate(ANIMALS)}
SPECIES=['bird','cat','deer','dog','frog','horse']
# ImageNet preprocessing (frozen backbone expects 224 + ImageNet norm)
tf=T.Compose([T.Resize(224),T.ToTensor(),T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
tr_full=torchvision.datasets.CIFAR10('./data',train=True,download=True,transform=tf)
te_full=torchvision.datasets.CIFAR10('./data',train=False,download=True,transform=tf)
def animal_subset(ds):
    idx=[i for i,t in enumerate(ds.targets) if t in REMAP]
    return idx
tr_idx=animal_subset(tr_full)[:6000]   # 6k for the linear probe (deterministic prefix)
te_idx=animal_subset(te_full)          # full animal test set, canonical order
print('train probe:',len(tr_idx),'  test:',len(te_idx))

In [ ]:
# frozen ImageNet ResNet18 as feature extractor (penultimate 512-d), NO training
bb=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
bb.fc=nn.Identity(); bb=bb.to(device).eval()
@torch.no_grad()
def extract(ds, idx):
    dl=DataLoader(torch.utils.data.Subset(ds,idx),128,shuffle=False,num_workers=2)
    feats=[]; labs=[]
    for x,y in dl:
        feats.append(bb(x.to(device)).cpu().numpy())
        labs.append(np.array([REMAP[int(t)] for t in y]))
    return np.concatenate(feats), np.concatenate(labs)
Xtr,ytr=extract(tr_full,tr_idx)
Xte,yte=extract(te_full,te_idx)
print('features:',Xtr.shape,Xte.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
probe=LogisticRegression(max_iter=2000,C=1.0).fit(Xtr,ytr)
pred=probe.predict(Xte)
print(f'Held-out accuracy: {accuracy_score(yte,pred):.4f}')
# Smart-Farmer census: per-species tally
counts=pd.Series(pred).value_counts().reindex(range(6),fill_value=0)
print('\nHerd census:')
for i,s in enumerate(SPECIES): print(f'  {s:6s}: {int(counts[i])}')

In [ ]:
pd.DataFrame({'id':range(len(pred)),'label':pred}).to_csv('submission.csv',index=False)
print('wrote submission.csv', len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)